# 04 - GUI / Present
Person 4. Loads model + present.csv, GUI to run predictions, exports final charts for the pptx team.

This is the ONLY notebook run live during the demo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import ipywidgets as widgets
from IPython.display import display

model = joblib.load('../model/attrition_model.pkl')
model_columns = joblib.load('../model/model_columns.pkl')
present_df = pd.read_csv('../data/present.csv')
present_df.head()


## Prep present set (no target column - unseen data)

In [ ]:
def prep(df):
    d = df.copy()
    cat_cols = d.select_dtypes(include='object').columns
    d = pd.get_dummies(d, columns=cat_cols, drop_first=True)
    d = d.reindex(columns=model_columns, fill_value=0)
    return d

X_present = prep(present_df)


## Run predictions

In [ ]:
preds = model.predict(X_present)
probs = model.predict_proba(X_present)[:, 1]

results = present_df.copy()
results['Predicted_Attrition'] = np.where(preds == 1, 'Yes', 'No')
results['Attrition_Risk_Score'] = probs.round(3)
results.head()


## Simple GUI - filter by department / risk threshold

In [ ]:
dept_col = 'Department' if 'Department' in results.columns else None

dept_options = ['All'] + sorted(results[dept_col].unique().tolist()) if dept_col else ['All']
dept_dropdown = widgets.Dropdown(options=dept_options, description='Dept:')
risk_slider = widgets.FloatSlider(value=0.5, min=0, max=1, step=0.05, description='Min risk:')
output = widgets.Output()

def update(change=None):
    output.clear_output()
    filtered = results.copy()
    if dept_col and dept_dropdown.value != 'All':
        filtered = filtered[filtered[dept_col] == dept_dropdown.value]
    filtered = filtered[filtered['Attrition_Risk_Score'] >= risk_slider.value]
    with output:
        display(filtered[[dept_col, 'Predicted_Attrition', 'Attrition_Risk_Score']] if dept_col else filtered[['Predicted_Attrition', 'Attrition_Risk_Score']])

dept_dropdown.observe(update, names='value')
risk_slider.observe(update, names='value')
update()

display(widgets.VBox([dept_dropdown, risk_slider, output]))


## Final chart - predicted attrition distribution (for pptx)

In [ ]:
plt.figure()
sns.countplot(x='Predicted_Attrition', data=results)
plt.title('Predicted Attrition - Present (New) Employees')
plt.savefig('../outputs/charts/predicted_attrition_distribution.png', bbox_inches='tight')
plt.show()


## Final chart - risk score distribution

In [ ]:
plt.figure()
sns.histplot(results['Attrition_Risk_Score'], bins=20, kde=True)
plt.title('Attrition Risk Score Distribution')
plt.savefig('../outputs/charts/risk_score_distribution.png', bbox_inches='tight')
plt.show()


## Export results for handoff

In [ ]:
results.to_csv('../outputs/present_predictions.csv', index=False)
print('charts + predictions saved to outputs/ - hand these to the pptx team')
